In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [3]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Forma de los datos de retornos: (16196, 23)


In [18]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

def construir_modelo_mixto(config, input_shape, n_assets=23):
    """Construye un modelo Híbrido: CNN -> LSTM -> Densa"""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # ---------------------------------------------------------
    # BLOQUE 1: EXTRACCIÓN DE PATRONES (CNN)
    # ---------------------------------------------------------
    # Usamos padding='same' para no reducir drásticamente la longitud en ventanas In:5
    model.add(Conv1D(filters=config['filtros_cnn'], 
                     kernel_size=config['kernel_size'], 
                     padding='same', 
                     activation='relu'))
    
    # Solo aplicamos MaxPooling si la ventana es de 10 días o más para no quedarnos sin datos
    if input_shape[0] >= 10:
        model.add(MaxPooling1D(pool_size=2))
        
    # ---------------------------------------------------------
    # BLOQUE 2: MEMORIA TEMPORAL (RNN)
    # ---------------------------------------------------------
    model.add(config['tipo_rnn'](config['neuronas_rnn'], return_sequences=False))
    
    # ---------------------------------------------------------
    # BLOQUE 3: TOMA DE DECISIONES (Densas)
    # ---------------------------------------------------------
    model.add(Dropout(config['dropout']))
    
    # Capa densa intermedia para procesar la salida de la LSTM
    if config['neuronas_densa'] > 0:
        model.add(Dense(config['neuronas_densa'], activation='relu'))
        
    # Salida: Regresión (23 activos)
    model.add(Dense(n_assets)) 
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    
    return model


def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [17]:
from tensorflow.keras.layers import LSTM, GRU

#  =====================================================================
# BANCOS DE PRUEBAS PARA REDES MIXTAS (CNN + RNN + Dense)
# =====================================================================
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]


# ----------------- IN: 5 DÍAS (Memoria Corta) -----------------
# El escáner (Kernel) debe ser minúsculo (2) porque solo hay 5 días.
hp_in5_corto = [
    # 1. El Freno de Mano Total: 
    # Reducimos la GRU al mínimo absoluto (4 neuronas) y subimos el Dropout a 0.4.
    # Es casi imposible memorizar ruido con solo 4 neuronas y un 40% de amnesia.
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': GRU,  'neuronas_rnn': 4,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Aprendiz Extra-Lento: 
    # Mantenemos 16 filtros y 8 neuronas, pero bajamos el LR a 0.0001 (1e-4) y Drop a 0.3.
    # Obligamos a la red a dar pasos milimétricos. Evitaremos el "efecto gancho" seguro.
    {'filtros_cnn': 16, 'kernel_size': 2, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.3, 'lr': 0.0001},
    
    # 3. El Filtro Extremo: 
    # Dropout al 50%. A cada paso, la red olvida la mitad de lo que ha visto.
    # Solo pasará la señal matemática si es escandalosamente obvia.
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005}
]

hp_in5_largo = [
    # 1. El Freno Extremo:
    # Quitamos la capa Densa (es un nido de overfitting para tan pocos datos).
    # Reducimos filtros y neuronas al mínimo. Dropout altísimo (0.4).
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': GRU,  'neuronas_rnn': 4,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Modelo Amnésico y Lento:
    # LSTM muy pequeña, kernel un poco más amplio (3 días), sin densa.
    # Dropout del 50% y un Learning Rate minúsculo. 
    # Obligamos a la red a no creerse nada de lo que ve a menos que sea una tendencia brutal.
    {'filtros_cnn': 8, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]


# ----------------- IN: 10 DÍAS (Memoria Quincenal) -----------------
# Aquí ya entra el MaxPooling que corta la secuencia a la mitad.
hp_in10_corto = [
    # 1. El "Ultra-Lento":
    # Bajamos los filtros y las neuronas a la mitad (8). Learning Rate bajísimo (1e-4).
    # Le costará horrores aprender, lo que evitará que la validación se dispare de golpe.
    {'filtros_cnn': 8,  'kernel_size': 3, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0001},

    # 2. El "Amnésico":
    # Mantenemos un poco de capacidad (16 filtros/neuronas) pero con un Dropout brutal (50%).
    # Olvidando la mitad de la información a cada paso, solo aprenderá tendencias reales.
    {'filtros_cnn': 16, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},

    # 3. El Escáner Semanal:
    # Aumentamos el Kernel a 5. La CNN mirará los 5 días de la semana de golpe 
    # en lugar de mirar de 3 en 3 días. Todo con máxima restricción.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005}
]

hp_in10_largo = [
    # 1. El Freno Extremo (Visión Semanal):
    # Mantenemos el escáner de 5 días, pero reducimos drásticamente los filtros 
    # y la memoria (8). Fuera la capa densa. Subimos el Dropout a 0.4.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Modelo Amnésico y Lento:
    # Le damos un poco más de filtros (16) para que busque más patrones, pero 
    # lo castigamos con un Dropout del 50% y un Learning Rate microscópico (1e-4).
    # O encuentra una tendencia brutalmente clara, o no aprenderá nada.
    {'filtros_cnn': 16, 'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

# ----------------- IN: 30 DÍAS (Memoria Mensual) -----------------
# Mucho ruido. Necesitamos un embudo estricto (Dropout alto).
hp_in30_corto = [
    # 1. El Escáner Conservador:
    # Kernel 5 (ve semanas). Pocos filtros (8) y memoria ligera (GRU 16).
    # Bajamos el LR a 0.0005 y subimos el Dropout a 0.4 para frenar el gancho suave.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Analista Lento:
    # Más filtros (16) y Kernel de 3 días. 
    # LR microscópico (1e-4) y Dropout altísimo (0.5). Obligamos a la LSTM a 
    # no precipitarse al predecir el día de mañana.
    {'filtros_cnn': 16, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

hp_in30_largo = [
    # 1. El Analista Macro (Moderado):
    # Mantenemos el escáner de semanas enteras (Kernel 5) pero bajamos la memoria a 16.
    # Fulminamos la capa Densa. Subimos el Dropout al 40%.
    {'filtros_cnn': 16, 'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Escéptico a Largo Plazo (Extremo):
    # Un poco más de filtros (32) para ver más variables, pero lo frenamos con 
    # un Dropout bestial del 50% y un LR muy bajo. 
    # O encuentra algo obvio, o se quedará plano (lo cual es mejor que sobreajustar).
    {'filtros_cnn': 32, 'kernel_size': 5, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

# ----------------- IN: 90 DÍAS (Memoria Trimestral) -----------------
# Riesgo inmenso de sobreajuste. 
hp_in90_corto = [
    # 1. El Compresor Extremo (Lento y seguro):
    # Mantenemos el Kernel de 10 (la CNN lee quincenas enteras, no días).
    # Bajamos los filtros a 8 y la GRU a 8. 
    # El LR baja a 0.0005 y el Dropout sube al 50%.
    # Queremos que la red ignore el 90% de la información irrelevante.
    {'filtros_cnn': 8, 'kernel_size': 10, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},
    
    # 2. El Escáner Semanal Asfixiado:
    # Kernel de 5 (ve semanas). 16 filtros y LSTM de 16.
    # Pero le metemos el freno de mano total: LR de 0.0001 (muy lento) y Dropout de 0.5.
    # Impedirá que la validación salga disparada en las primeras 10 épocas.
    {'filtros_cnn': 16, 'kernel_size': 5,  'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

hp_in90_largo = [
    # 1. El Macro-Analista Estricto:
    # Mantenemos el Kernel de 10 (la CNN lee los datos en bloques de dos semanas).
    # Bajamos los filtros y la memoria a 16. ¡Fuera la capa Densa!
    # Dropout máximo (0.5) para que no se memorice el ruido diario.
    {'filtros_cnn': 16, 'kernel_size': 10, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},
    
    # 2. El Observador Trimestral Lento:
    # 32 filtros (lo máximo absoluto que le vamos a permitir) y Kernel 5.
    # LSTM de 16 neuronas sin capa densa. 
    # LR bajísimo (0.0001) para que los ajustes sean milimétricos.
    {'filtros_cnn': 32, 'kernel_size': 5,  'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]


In [23]:
# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_mixtas = np.zeros((len(input_windows), len(output_windows))) 
matriz_mae_val_mixtas = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_test_mixtas = np.zeros((len(input_windows), len(output_windows)))


# Matrices Baselines Validacion
matriz_mae_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_sma_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_bh_val = np.zeros((len(input_windows), len(output_windows)))


# Matrices Baselines Test
matriz_mae_naive_test = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_sma_test = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_bh_test = np.zeros((len(input_windows), len(output_windows)))

# Aumentamos la paciencia a 15 épocas
early_stop = EarlyStopping(
    monitor ='val_loss', 
    patience = 20,               # <--- CAMBIO AQUÍ
    restore_best_weights = True  # IMPORTANTE: Que devuelva los pesos de la mejor época
)

In [24]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================


print("\nIniciando entrenamiento de modelos...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación CRONOLÓGICA: 80% Train, 10% Validacion, 10% Test
        # split_1 = int(len(X) * 0.8)
        # split_2 = int(len(X) * 0.9)
        
        # Para un esquema 70% Train, 20% Validacion, 10% Test
        split_1 = int(len(X) * 0.70) # Aquí cortamos el Train
        split_2 = int(len(X) * 0.90) # Aquí cortamos la Validación (del 70% al 90% = 20%)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        
        # =====================================================================
        # 3. Baselines (AHORA EN VALIDACIÓN Y TEST)
        # =====================================================================

        # Calcular Baselines en Validación
        mae_naive_val, mae_sma_val, mae_bh_val = calcular_baselines(X_val, y_val, y_train_mean)
        matriz_mae_naive_val[i, j] = mae_naive_val
        matriz_mae_sma_val[i, j] = mae_sma_val
        matriz_mae_bh_val[i, j] = mae_bh_val
        
        # Calcular Baselines en Test
        mae_naive_test, mae_sma_test, mae_bh_test = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive_test[i, j] = mae_naive_test
        matriz_mae_sma_test[i, j] = mae_sma_test
        matriz_mae_bh_test[i, j] = mae_bh_test
        

        print("--- Baselines VALIDACIÓN ---")
        print(f"Naive: {mae_naive_val:.6f} | SMA: {mae_sma_val:.6f} | Buy&Hold: {mae_bh_val:.6f}")
        print("--- Baselines TEST ---")
        print(f"Naive: {mae_naive_test:.6f} | SMA: {mae_sma_test:.6f} | Buy&Hold: {mae_bh_test:.6f}\n")


        # 4. Búsqueda del mejor modelo recurrente
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None


        # DIVIDE Y VENCERÁS: Selección de hiperparámetros

        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "Mixto In:5 Corto" if out_w in [1, 5] else "Mixto In:5 Largo"
            
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "Mixto In:10 Corto" if out_w in [1, 5] else "Mixto In:10 Largo"
            
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "Mixto In:30 Corto" if out_w in [1, 5] else "Mixto In:30 Largo"
            
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "Mixto In:90 Corto" if out_w in [1, 5] else "Mixto In:90 Largo"
            
        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            nombre_rnn = config['tipo_rnn'].__name__
            print(f" -> Entrenando Mixto: CNN({config['filtros_cnn']}F, K{config['kernel_size']}) + {nombre_rnn}({config['neuronas_rnn']}) + Densa({config['neuronas_densa']})")
            
            # Llamamos a TU función de arquitectura mixta
            modelo = construir_modelo_mixto(config, input_shape=(in_w, 23))
            
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0)
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n -> Ganador Mixto: CNN({mejor_config['filtros_cnn']}F, K{mejor_config['kernel_size']}) + {nombre_rnn}({mejor_config['neuronas_rnn']}) + Densa({mejor_config['neuronas_densa']})")
        
        # 5. Evaluación final del GANADOR en TRAIN, VALIDACIÓN y TEST
        mae_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        mae_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        # Guardar en sus respectivas matrices (¡recuerda inicializarlas antes del bucle!)
        matriz_mae_train_mixtas[i, j] = mae_train_ganador
        matriz_mae_val_mixtas[i, j] = mae_val_ganador
        matriz_mae_test_mixtas[i, j] = mae_test_ganador
        
        print(f"MAE del Modelo Ganador en TRAIN:      {mae_train_ganador:.6f}")
        print(f"MAE del Modelo Ganador en VALIDACIÓN: {mae_val_ganador:.6f}")
        print(f"MAE del Modelo Ganador en TEST:       {mae_test_ganador:.6f}")
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(12, 6)) # Hacemos la gráfica un poco más ancha para el título largo
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Extraer los hiperparámetros de la arquitectura Mixta
        f_cnn = mejor_config['filtros_cnn']
        k_cnn = mejor_config['kernel_size']
        nombre_rnn = mejor_config['tipo_rnn'].__name__
        n_rnn = mejor_config['neuronas_rnn']
        n_densa = mejor_config['neuronas_densa']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        # Construir un título que resuma la arquitectura completa
        titulo_arqui = f"CNN({f_cnn}F, K{k_cnn}) + {nombre_rnn}({n_rnn}) + Densa({n_densa})"
        plt.title(f"Convergencia Mixta: {titulo_arqui}\nLR: {l_rate} | Drop: {d_out} | (Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen en una nueva carpeta para no pisar las viejas
        os.makedirs('graficas_convergencia_mixtas', exist_ok=True)
        nombre_archivo = f"graficas_convergencia_mixtas/conver_mixto_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
--- Baselines VALIDACIÓN ---
Naive: 0.015404 | SMA: 0.011784 | Buy&Hold: 0.010569
--- Baselines TEST ---
Naive: 0.017825 | SMA: 0.013638 | Buy&Hold: 0.012268

 -> Usando banco de pruebas: [Mixto In:5 Corto]
 -> Entrenando Mixto: CNN(8F, K2) + GRU(4) + Densa(0)
 -> Entrenando Mixto: CNN(16F, K2) + LSTM(8) + Densa(0)
 -> Entrenando Mixto: CNN(8F, K2) + LSTM(8) + Densa(0)

 -> Ganador Mixto: CNN(8F, K2) + LSTM(8) + Densa(0)
MAE del Modelo Ganador en TRAIN:      0.011832
MAE del Modelo Ganador en VALIDACIÓN: 0.010574
MAE del Modelo Ganador en TEST:       0.012274

 Ventana Entrada: 5 días | Ventana Salida: 5 días
--- Baselines VALIDACIÓN ---
Naive: 0.011804 | SMA: 0.006917 | Buy&Hold: 0.004728
--- Baselines TEST ---
Naive: 0.013683 | SMA: 0.008049 | Buy&Hold: 0.005594

 -> Usando banco de pruebas: [Mixto In:5 Corto]
 -> Entrenando Mixto: CNN(8F, K2) + GRU(4) + Densa(0)
 -> Entrenando Mixto: CNN(16F, K

In [25]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)")
print("="*50)
df_mixtas_train = pd.DataFrame(matriz_mae_train_mixtas, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_mixtas_train)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)")
print("="*50)
df_mixtas_val = pd.DataFrame(matriz_mae_val_mixtas, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_mixtas_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN TEST (RNN)")
print("="*50)
df_mixtas_test = pd.DataFrame(matriz_mae_test_mixtas, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_mixtas_test)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)")
print("="*50)
df_naive_test = pd.DataFrame(matriz_mae_naive_test, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive_test)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (VALIDACION)")
print("="*50)
df_naive_val = pd.DataFrame(matriz_mae_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (TEST)")
print("="*50)
df_sma_test = pd.DataFrame(matriz_mae_sma_test, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma_test)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (VALIDACION)")
print("="*50)
df_sma_val = pd.DataFrame(matriz_mae_sma_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh_test = pd.DataFrame(matriz_mae_bh_test, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh_test)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (VALIDACION)")
print("="*50)
df_bh_val = pd.DataFrame(matriz_mae_bh_val, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh_val)



MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.011832  0.005508  0.002216  0.001272
In_10  0.012125  0.006145  0.002884  0.002250
In_30  0.012052  0.005763  0.002295  0.002073
In_90  0.012022  0.005860  0.002628  0.001332

MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.010574  0.004736  0.001936  0.001106
In_10  0.010802  0.005442  0.002541  0.002059
In_30  0.010775  0.004989  0.001996  0.002053
In_90  0.010751  0.005180  0.002375  0.001176

MATRIZ DE RESULTADOS FINALES EN TEST (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.012274  0.005599  0.002322  0.001270
In_10  0.012522  0.006227  0.003056  0.002366
In_30  0.012460  0.005869  0.002459  0.002111
In_90  0.012426  0.006002  0.002874  0.001357

MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)
          Out_1     Out_5    Out_30    Out_90
In_5   0.017825  0.013683  0.012529  0.012231
In_10  0.017831  0.013683 